In [ ]:
import torch
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from mapie.regression import SplitConformalRegressor
from mapie.conformity_scores import MultivariateResidualNormalisedScore


In [ ]:
class Generator2D:
    def __init__(self, f, matrix_transform, noise_type='gaussian', noise_std=1.0):
        """
        Args:
            f (callable): Function f(X) -> (N, 2) (The conditional mean)
            matrix_transform (callable): Function A(X) -> (N, 2, 2). 
                                         Defines local noise shape and orientation.
            noise_type (str): 'gaussian', 'uniform', or 'exponential'
            noise_std (float): Global noise scaling factor
        """
        self.f = f
        self.matrix_transform = matrix_transform
        self.noise_type = noise_type
        self.noise_std = noise_std

    def _get_noise(self, n):
        """Generates a raw noise vector of shape (n, 2, 1)"""
        if self.noise_type == 'gaussian':
            noise = torch.randn(n, 2)
        elif self.noise_type == 'uniform':
            # Scale to [-1, 1] range
            noise = torch.rand(n, 2) * 2 - 1
        elif self.noise_type == 'exponential':
            noise = torch.distributions.Exponential(rate=1.0).sample((n, 2))
        else:
            raise ValueError(f"Unknown noise type: {self.noise_type}")
        
        # Reshape for matrix multiplication: (N, 2) -> (N, 2, 1)
        return (noise * self.noise_std).unsqueeze(2)

    def generate(self, n):
        """
        Generates n samples based on:
        X ~ U[-1, 1]
        Y = f(X) + A(X) @ Noise
        """
        # 1. Sample X from a uniform distribution [-1, 1]
        x = 2 * torch.rand(n, 1) - 1

        # 2. Compute conditional mean f(X) -> (n, 2)
        fx = self.f(x)

        # 3. Compute transformation matrix (root covariance) A(X) -> (n, 2, 2)
        A_x = self.matrix_transform(x)

        # 4. Generate raw noise -> (n, 2, 1)
        noise = self._get_noise(n)

        # 5. Apply transformation: (n, 2, 2) @ (n, 2, 1) -> (n, 2, 1)
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)

        # 6. Final output: Y = mean + correlated noise
        y = fx + correlated_noise

        return x.numpy(), y.numpy()

    def generate_specific_y_given_x(self, x_tensor, n=1):
        """Returns n samples of Y for every provided X in x_tensor"""
        # Repeat X values to handle vectorization for multiple samples per input
        x_repeated = x_tensor.repeat_interleave(n, dim=0)
        
        fx = self.f(x_repeated)
        A_x = self.matrix_transform(x_repeated)
        noise = self._get_noise(x_repeated.shape[0])
        
        # Apply transformation and squeeze to (Total_N, 2)
        correlated_noise = torch.bmm(A_x, noise).squeeze(2)
        y_flat = fx + correlated_noise
        
        return y_flat
    
def matrix_transform_function(x):
    """
    Example transformation function that creates a 2x2 matrix 
    dependent on the input x.
    """
    n = x.shape[0]
    x_sq = x.squeeze()
    
    # Initialize identity matrices
    matrices = torch.eye(2).unsqueeze(0).repeat(n, 1, 1)
    
    # Dynamically adjust matrix components based on x
    matrices[:, 0, 0] = x_sq**2 + 0.5 
    matrices[:, 0, 1] = torch.sin(x_sq * 2)
    matrices[:, 1, 1] = torch.abs(x_sq) + 0.2

    return matrices

def circle_f(x):
    """Computes a mean vector that follows a circular/trigonometric path"""
    return torch.cat([torch.sin(x * 3), torch.cos(x * 3)], dim=1)

def get_ellipse_points(model, x_val, level):
    """Return ellipse points for a model at a fixed x value."""
    x_tensor = np.array([[x_val]])
    center, sigma = model.get_distribution(x_tensor)
    mu = center[0]
    cov = sigma[0]

    L = np.linalg.cholesky(cov)
    theta = np.linspace(0, 2 * np.pi, 100)
    circle_points = np.stack([np.cos(theta), np.sin(theta)], axis=0)
    return mu[:, None] + L @ (level * circle_points)

In [ ]:
n_train = 20_000
n_test = 1_000
n_calibration = 1_000
n_stop = 1_000

dtype = torch.float32


# Initialize the generator with Exponential noise
generator = Generator2D(
    f=circle_f,
    matrix_transform=matrix_transform_function, 
    noise_std=1.0,
    noise_type="exponential"
)

x_samples, y_samples = generator.generate(100)

x_train, y_train = generator.generate(n_train)
x_stop, y_stop = generator.generate(n_stop)
x_calibration, y_calibration = generator.generate(n_calibration)
x_test, y_test = generator.generate(n_test)

# Without MAPIE

In [ ]:

class DummyTrainer:
    """A mock estimator to simulate the behavior of Trainer in unit tests. This trainer does not have a get_standardized_score function."""
    def __init__(self, covariance_trainer, output_dim):
        self.covariance_trainer = covariance_trainer
        self.output_dim = output_dim

    def fit(self, X, y, y_pred, **kwargs):
        self.fitted = True
        return self
    
    def predict(self, X):
        return self.covariance_trainer.predict(X)

    def get_distribution(self, X):
        n_samples = X.shape[0]
        # Returns dummy predictions and an identity covariance matrix per sample
        y_pred = self.predict(X)
        sigma = np.array([np.eye(self.output_dim) for _ in range(n_samples)])
        return y_pred, sigma

    def get_covariance_matrix(self, X):
        n_samples = X.shape[0]
        # Returns an identity covariance matrix per sample
        return np.array([np.eye(self.output_dim) for _ in range(n_samples)])
  

In [ ]:
calibrator = MultivariateResidualNormalisedScore()
calibrator.fit(x_train, y_train, num_epochs=20)
scores = calibrator.get_signed_conformity_scores(y_calibration, None, X=x_calibration)

n = len(scores)
alpha = 0.1
p = int(np.ceil((n+1)*(1-alpha)))
if p > n:
    raise ValueError("The number of calibration samples is too low to reach the desired alpha level.")
y_pred = calibrator.predict(x_train)
y_pred_cal = calibrator.predict(x_calibration)

scores_sorted = np.sort(scores)
level = scores_sorted[p]
print(level)

calibrator_dummy = MultivariateResidualNormalisedScore(covariance_estimator=DummyTrainer(calibrator.covariance_estimator_, 2) )
calibrator_dummy.fit(x_train, y_train, y_pred=y_pred, num_epochs=0)

In [ ]:
FS = 18
fig = plt.figure(figsize=(8, 8), dpi=120)
ax = fig.add_subplot(111, projection='3d')

x_range = (-0.6, 1)
num_slices = 10
n_samples = 100
xs = np.linspace(x_range[0], x_range[1], num_slices)

models_and_colors = [
    (calibrator.covariance_estimator_, 'green'),
    (calibrator_dummy.covariance_estimator_, 'red'),
]

for x_val in xs:
    # Plot ellispes
    for model, color in models_and_colors:
        ellipse_points = get_ellipse_points(model, x_val, level)
        ax.plot(
            np.full_like(ellipse_points[0, :], x_val),
            ellipse_points[0, :],
            ellipse_points[1, :],
            color=color,
            linewidth=2,
            alpha=0.8,
            zorder=2,
        )

    # Plot samples
    samples = generator.generate_specific_y_given_x(
        torch.tensor([x_val], dtype=torch.float32).unsqueeze(0),
        n=n_samples,
    )
    samples = samples.detach().cpu().numpy()
    ax.scatter(
        np.full(n_samples, x_val),
        samples[:, 0],
        samples[:, 1],
        c='grey',
        marker='o',
        s=5,
        alpha=0.3,
        zorder=1,
    )

ax.set_xlabel(r'$X$', fontsize=FS, labelpad=15)
ax.set_ylabel(r'$Y_1$', fontsize=FS, labelpad=15)
ax.text(0.75, 16, 5, r'$Y_2$', fontsize=FS, fontweight='bold', ha='center')

ax.set_xlim(x_range[0], x_range[1])
ax.set_zlim(-7, 7)

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.tick_params(axis='both', which='major', labelsize=FS)

ax.view_init(elev=20, azim=-60)
plt.subplots_adjust(left=0.0, right=0.95, bottom=0.05, top=0.95)
plt.show()

# with MAPIE

In [ ]:
score = MultivariateResidualNormalisedScore(mode="low_rank")
score.fit(x_train, y_train, num_epochs=20)

confidence_level = 0.9
mapie_stand = SplitConformalRegressor(
    estimator=score.covariance_estimator_,
    conformity_score=score,
    confidence_level=confidence_level, 
    prefit=True
)
mapie_stand.conformalize(x_calibration, y_calibration)

In [ ]:
# conformalization
scores = mapie_stand._mapie_regressor.conformity_scores_
n = len(scores)
alpha = mapie_stand._alphas[0]
p = int(np.ceil((n+1)*(1-alpha)))
if p > n:
    raise ValueError("The number of calibration samples is too low to reach the desired alpha level.")
scores_sorted = np.sort(scores)
level = scores_sorted[p]
print(level)

In [ ]:
score_dummy = MultivariateResidualNormalisedScore(covariance_estimator=DummyTrainer(score.covariance_estimator_, 2))
score_dummy.fit(x_train, y_train, y_pred=y_pred, num_epochs=0)

mapie_stand_dummy = SplitConformalRegressor(
    estimator=score_dummy.covariance_estimator_,
    conformity_score = score_dummy,
    confidence_level=confidence_level, 
    prefit=True
)

In [ ]:
FS = 18
fig = plt.figure(figsize=(8, 8), dpi=120)
ax = fig.add_subplot(111, projection='3d')

x_range = (-0.6, 1)
num_slices = 10
n_samples = 100
xs = np.linspace(x_range[0], x_range[1], num_slices)

models_and_colors = [
    (mapie_stand._conformity_score.covariance_estimator_, 'green'),
    (mapie_stand_dummy._conformity_score.covariance_estimator_, 'red'),
]

for x_val in xs:
    # Plot ellispes
    for model, color in models_and_colors:
        ellipse_points = get_ellipse_points(model, x_val, level)
        ax.plot(
            np.full_like(ellipse_points[0, :], x_val),
            ellipse_points[0, :],
            ellipse_points[1, :],
            color=color,
            linewidth=2,
            alpha=0.8,
            zorder=2,
        )

    # Plot samples
    samples = generator.generate_specific_y_given_x(
        torch.tensor([x_val], dtype=torch.float32).unsqueeze(0),
        n=n_samples,
    )
    samples = samples.detach().cpu().numpy()
    ax.scatter(
        np.full(n_samples, x_val),
        samples[:, 0],
        samples[:, 1],
        c='grey',
        marker='o',
        s=5,
        alpha=0.3,
        zorder=1,
    )

ax.set_xlabel(r'$X$', fontsize=FS, labelpad=15)
ax.set_ylabel(r'$Y_1$', fontsize=FS, labelpad=15)
ax.text(0.75, 16, 5, r'$Y_2$', fontsize=FS, fontweight='bold', ha='center')

ax.set_xlim(x_range[0], x_range[1])
ax.set_zlim(-7, 7)

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.tick_params(axis='both', which='major', labelsize=FS)

ax.view_init(elev=20, azim=-60)
plt.subplots_adjust(left=0.0, right=0.95, bottom=0.05, top=0.95)
plt.show()